
## Folder 02 — CSV Upload and Source Validation:


**First, upload your seven CSV files into the Databricks Volume using this structure:**


## 1. Pipeline Overview:


Your pipeline follows this sequence:


### Seven Source CSV Files:


1. Customers
2. Products
3. Orders
4. Order Items
5. Payments
6. Shipments
7. Inventory


### Landing Zone:


Store uploaded source CSV files in Databricks Unity Catalog Volume.


### Source Validation:


**Perform the following checks:**

- File existence
- Schema validation
- Record count validation
- Null value detection
- Missing primary key detection
- Duplicate record detection
- File readability


### Validation Report


Save validation results as a Delta table for auditing and monitoring.


### Pipeline Gate:




| Validation Status | Pipeline Action |
|---|---|
| PASS | Continue to Bronze layer |
| FAIL | Stop the pipeline |


## 2. Source Validation Workflow:


```text
       Seven Source CSV Files
                 |
                 v
            Landing Zone
                 |
                 v
          Source Validation
                 |
                 v
          Validation Report
                 |
                 v
           Pipeline Gate
                 |
          +------+------+
          |             |
          v             v
         PASS          FAIL
          |             |
          v             v
     Bronze Layer   Stop Pipeline
          |
          v
     Continue ETL


## PySpark Libraries and Data Types:


| Library | Purpose |
|---|---|
| `pyspark.sql.functions` | Performs DataFrame operations |
| `StructType` | Defines a complete schema |
| `StructField` | Defines individual columns |
| `StringType` | Defines string columns |
| `LongType` | Defines large integer columns |

In [0]:
 ## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType
)

print("Libraries imported successfully.")



## 2. Project configuration

CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "data_volume"

BASE_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/ecommerce_lakehouse"
)

LANDING_PATH = f"{BASE_PATH}/landing"

VALIDATION_REPORT_TABLE = (
    f"{CATALOG}.{SCHEMA}.source_validation_report"
)

DATASETS = [
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"
]

PRIMARY_KEYS = {
    "customers": ["customer_id"],
    "products": ["product_id"],
    "orders": ["order_id"],
    "order_items": ["order_item_id"],
    "payments": ["payment_id"],
    "shipments": ["shipment_id"],
    "inventory_snapshots": ["inventory_snapshot_id"]
}

EXPECTED_COLUMNS = {
    "customers": [
        "customer_id", "first_name", "last_name", "email",
        "phone", "city", "state", "created_at"
    ],

    "products": [
        "product_id", "product_name", "category", "unit_price",
        "unit_cost", "stock_status", "created_at"
    ],

    "orders": [
        "order_id", "customer_id", "order_timestamp",
        "order_status", "currency", "order_total", "sales_channel"
    ],

    "order_items": [
        "order_item_id", "order_id", "product_id", "quantity",
        "unit_price", "discount_rate", "line_total"
    ],

    "payments": [
        "payment_id", "order_id", "payment_method",
        "payment_status", "amount", "payment_timestamp",
        "transaction_id"
    ],

    "shipments": [
        "shipment_id", "order_id", "carrier", "shipment_status",
        "shipped_date", "delivered_date", "tracking_number"
    ],

    "inventory_snapshots": [
        "inventory_snapshot_id", "product_id",
        "quantity_on_hand", "snapshot_date", "warehouse_id"
    ]
}

print("Project configuration loaded.")



## 3. Check source CSV files

source_files = {}

for dataset in DATASETS:

    file_path = f"{LANDING_PATH}/{dataset}/{dataset}.csv"

    try:
        dbutils.fs.ls(file_path)
        source_files[dataset] = file_path
        print(f"FOUND: {dataset}.csv")

    except Exception as e:
        source_files[dataset] = None
        print(f"MISSING: {dataset}.csv")
        print(str(e)[:300])

print("Source file availability check completed.")


## 4. Define validation reusable function

def validate_source_csv(dataset, file_path):

    expected_columns = EXPECTED_COLUMNS[dataset]
    primary_key = PRIMARY_KEYS[dataset]

    # Read CSV with header and inferred data types

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "PERMISSIVE")
        .option("multiLine", "true")
        .load(file_path)
    )

    # Normalize column names by removing surrounding spaces

    df = df.toDF(*[column.strip() for column in df.columns])

    '''
        df.columns → returns all column names.
        column.strip() → removes surrounding whitespace.
        * → unpacks the generated list of column names.
        toDF() → renames DataFrame columns.
    '''

    actual_columns = df.columns

    missing_columns = [
        column for column in expected_columns
        if column not in actual_columns
    ]

    unexpected_columns = [
        column for column in actual_columns
        if column not in expected_columns
    ]

    '''
        Expected:

            customer_id
            first_name
            last_name
            email
            phone
            city
            state

        created_at:

            Actual:
            customer_id
            first_name
            last_name
            email
            city
            state
            created_at

        Output:

            ["phone"]
            The phone column is missing.
    '''

    # Record count

    total_records = df.count()



    # Count missing or blank values

    missing_value_expressions = [
        F.sum(
            F.when(
                F.col(column).isNull() |
                (F.trim(F.col(column).cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias(column)
        for column in actual_columns
    ]

    '''
        A. F.col(column):

            References a DataFrame column.

        B. .isNull():

            Checks whether a value is NULL.

        C. .cast("string"):

            Converts the value into a string for checking.

        D. F.trim():

            Removes leading and trailing spaces.

        E. == "" :

            Checks whether the value is empty.

        F. | :

            Represents OR between two Spark column conditions.

        G. F.when():

            Implements conditional logic.

            F.when(condition, 1).otherwise(0)

            If the value is missing, return 1. Otherwise, return 0.

        H. F.sum():

            Adds the missing-value indicators for each column.

        I. .alias(column):

            Assigns the resulting aggregate its column name.
    '''

    missing_value_result = df.agg(
        *missing_value_expressions
    ).first().asDict()

    '''
     This executes the aggregation and retrieves the result as a Python dictionary.

     Example:
        {       
            "customer_id": 0,    
            "first_name": 12,    
            "last_name": 5,    
            "email": 20,    
            "phone": 30,    
            "city": 0,    
            "state": 0,    
            "created_at": 0
        }
    '''

    total_missing_values = sum(
        int(value or 0)
        for value in missing_value_result.values()
    )

    '''
        This adds the missing values across every column.

            For the example above:
            
                12+5+20+30=67
            

            Result:

                total_missing_values = 67


            Important: 
            
            This is the total number of missing cells, not the number of rows containing at least one missing value.
    '''

    # Count rows with missing primary keys

    available_keys = [
        key for key in primary_key
        if key in actual_columns
    ]

    '''
        This checks whether the primary key columns exist in the actual DataFrame.

            For customers:
                customer_id
    '''

    if len(available_keys) == len(primary_key):

            missing_key_condition = None
        
    '''  
        If all expected primary key columns are available, the following code executes:
    '''

        for key in primary_key:

            condition = (
                F.col(key).isNull() |
                (F.trim(F.col(key).cast("string")) == "")
            )

            '''
                Checks whether the primary key contains NULL or blank values.
            '''

            missing_key_condition = (
                condition
                if missing_key_condition is None
                else missing_key_condition | condition
            )

        missing_primary_key_records = (
            df.filter(missing_key_condition).count()
        )

        '''
            This counts records with missing primary keys.
        '''

        # Count duplicate primary-key groups

        duplicate_primary_key_groups = (
            df.groupBy(*primary_key)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        '''
            A. groupBy(*primary_key):

                Groups records using the primary key.

            B. .count():

                Counts records within each group.

            C. .filter(F.col("count") > 1):

                Keeps only groups containing more than one record.

            D. Final .count():

                Counts the number of duplicate groups.
        '''

    else:
        missing_primary_key_records = -1
        duplicate_primary_key_groups = -1

    '''
        If the required primary key column is missing, the code cannot perform primary-key validation.
        Therefore, it uses -1 as a marker for unavailable validation.
        It does not mean there are negative records.
    '''

    # Structural validation

    structure_status = (
        "PASS" if len(missing_columns) == 0 else "FAIL"
    )


    result = {
        "dataset": dataset,
        "file_path": file_path,
        "total_records": total_records,
        "total_columns": len(actual_columns),
        "missing_columns": ", ".join(missing_columns),
        "unexpected_columns": ", ".join(unexpected_columns),
        "total_missing_values": total_missing_values,
        "missing_primary_key_records": missing_primary_key_records,
        "duplicate_primary_key_groups": duplicate_primary_key_groups,
        "structure_status": structure_status
    }

    return result


## 5. Validate all seven datasets

validation_results = []

for dataset in DATASETS:

    file_path = source_files[dataset]

    if file_path is None:

        validation_results.append({
            "dataset": dataset,
            "file_path": "NOT FOUND",
            "total_records": 0,
            "total_columns": 0,
            "missing_columns": "SOURCE FILE MISSING",
            "unexpected_columns": "",
            "total_missing_values": 0,
            "missing_primary_key_records": -1,
            "duplicate_primary_key_groups": -1,
            "structure_status": "FAIL"
        })

        continue

    try:

        result = validate_source_csv(dataset, file_path)

        validation_results.append(result)

        print(
            f"{dataset}: "
            f"{result['structure_status']} | "
            f"Records: {result['total_records']} | "
            f"Duplicate key groups: "
            f"{result['duplicate_primary_key_groups']}"
        )

    except Exception as e:

        validation_results.append({
            "dataset": dataset,
            "file_path": file_path,
            "total_records": 0,
            "total_columns": 0,
            "missing_columns": str(e)[:500],
            "unexpected_columns": "",
            "total_missing_values": 0,
            "missing_primary_key_records": -1,
            "duplicate_primary_key_groups": -1,
            "structure_status": "FAIL"
        })

        print(f"ERROR validating {dataset}: {str(e)}")

print("All source validation attempts completed.")



## 6. Create validation report

report_schema = StructType([
    StructField("dataset", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("total_records", LongType(), True),
    StructField("total_columns", LongType(), True),
    StructField("missing_columns", StringType(), True),
    StructField("unexpected_columns", StringType(), True),
    StructField("total_missing_values", LongType(), True),
    StructField("missing_primary_key_records", LongType(), True),
    StructField("duplicate_primary_key_groups", LongType(), True),
    StructField("structure_status", StringType(), True)
])

validation_report = spark.createDataFrame(
    validation_results,
    schema=report_schema
)

display(validation_report)



## 7. Save validation report as Delta table

(
    validation_report.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(VALIDATION_REPORT_TABLE)
)

print(f"Validation report saved: {VALIDATION_REPORT_TABLE}")



## 8. Display source validation summary

validation_report.createOrReplaceTempView(
    "source_validation_summary"
)

display(
    spark.sql("""
        SELECT
            dataset,
            total_records,
            total_columns,
            total_missing_values,
            missing_primary_key_records,
            duplicate_primary_key_groups,
            structure_status
        FROM source_validation_summary
        ORDER BY dataset
    """)
)



## 9. Stop pipeline if a source file or required column is missing

failed_datasets = [
    record["dataset"]
    for record in validation_report.collect()
    if record["structure_status"] == "FAIL"
]

if failed_datasets:

    raise Exception(
        "Source validation failed for: "
        + ", ".join(failed_datasets)
    )

print("All seven source files passed structural validation.")



## 10. Final validation result

total_datasets = validation_report.count()

total_source_records = (
    validation_report
    .agg(F.sum("total_records")
    .alias("total"))
    .first()["total"] or 0
)

print("=" * 60)
print("SOURCE VALIDATION SUMMARY")
print("=" * 60)

print(f"Datasets validated : {total_datasets}")
print(f"Total source rows  : {total_source_records}")
print(f"Report table       : {VALIDATION_REPORT_TABLE}")

print("=" * 60)
print("SOURCE STRUCTURE VALIDATION COMPLETED")
print("=" * 60)
